In [127]:
patch_drop_block_number=[]
# average_percent_dro_embedding = 50
drop_rate_layers=0.00

In [128]:
import matplotlib.pyplot as plt
from time import time
import numpy as np
import os
import cv2
import json
import torch
from timm.models.vision_transformer import VisionTransformer, PatchEmbed
import torch
import torch.nn as nn
import numpy as np
import os
import re
import timm

In [129]:
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
img_folder = "/home/viraj/my_project/val/val2/"
txt_path=f"vit_tiny_similarity_score_with_cls_token_accuracy_drop_0.25__percent_on_{patch_drop_block_number}.txt"
json_file = "/home/viraj/my_project/imagenet_class_index.json"
# line_r=0
 

In [130]:
# # pretrained_vit = vit_large_patch16_224(pretrained=True).to(device)

# pretrained_vit = timm.create_model('vit_tiny_patch16_224', pretrained=True).to(device)

import torch
import timm


# 1. Set up device
pretrained_vit = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Load the pretrained distilled DeiT-Tiny model
pretrained_vit = timm.create_model(
    'deit_tiny_distilled_patch16_224.fb_in1k',
    pretrained=True
).to(device)

pretrained_vit.eval()

VisionTransformerDistilled(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=192, out_features=576, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=192, out_features=192, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=192, out_features=768, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)
        (norm): Id

In [131]:
def plot(image):
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.show()

In [132]:
def copy_image(img):
    """
    Create a copy of the input image without using any libraries.

    Args:
    - img: Input image (list of lists representing pixel values)

    Returns:
    - img_copy: A copy of the input image
    """
    # Determine the dimensions of the image
    height = len(img)
    width = len(img[0])

    # Create an empty list to store the copied image
    img_copy = []

    # Iterate over each row of the image
    for i in range(height):
        # Create a copy of the current row
        row_copy = img[i][:]  # or row_copy = list(img[i]) for deep copy
        # Append the copied row to the copied image
        img_copy.append(row_copy)

    return img_copy

In [133]:
def bgr_to_rgb(img):
    """
    Convert an image from BGR to RGB color space.

    Args:
    - img: Input image in BGR format (numpy array)

    Returns:
    - img_rgb: Image converted to RGB color space
    """
    # Reverse the order of color channels
    img_rgb = img[:, :, ::-1]

    return img_rgb

In [134]:
def resize_nearest_neighbor(image, size):
    original_height = len(image)
    original_width = len(image[0])
    new_width, new_height = size

    if isinstance(image[0][0], list) or isinstance(image[0][0], np.ndarray):
        # For color images
        resized_image = np.zeros((new_height, new_width, 3), dtype=np.uint8)
        for i in range(new_height):
            for j in range(new_width):
                # Calculate the nearest pixel in the original image
                x = int(j * original_width / new_width)
                y = int(i * original_height / new_height)
                resized_image[i, j] = image[y][x]
    else:
        # For grayscale images
        resized_image = np.zeros((new_height, new_width), dtype=np.uint8)
        for i in range(new_height):
            for j in range(new_width):
                # Calculate the nearest pixel in the original image
                x = int(j * original_width / new_width)
                y = int(i * original_height / new_height)
                resized_image[i, j] = image[y][x]
    
    return np.array(resized_image)

In [135]:
def save_average_percentage(file_path, average_percent, accuracy, processed_image, w):
    with open(file_path, 'a') as file:
        file.write(f"Epoch: Average percentage_patch_drop: {average_percent}, Accuracy: {accuracy}, processed_image: {processed_image}, scan: {w}\n")

In [136]:
def load_mapping_file(json_file):
    with open(json_file, 'r') as f:
        mapping_data = json.load(f)
    return mapping_data

In [137]:
import torchvision.transforms as transforms
def transform_image(image, device):

    image=transforms.ToTensor()(image)
    image= image.unsqueeze(0)
    image=image.to(device)    
    

    
    return image

In [138]:
mapping_data = load_mapping_file(json_file)

def get_key_by_value(predicted_index, class_name):
    
    for key, values in mapping_data.items():
        key = int(key)
        
        if class_name in values:
            # print(key, predicted_index, "key and predicted_index")
            # print(f"Found class_name '{class_name}' in values: {values}")
            if int(key) == predicted_index:
                return True
    return False

In [139]:
patch_drop_percentage = []
def average_patch_drop(mask):
    count_ones = mask.count(1)
    count_ones=count_ones-1
    y = (((196 - count_ones ) / 196) * 100)
    patch_drop_percentage.append(y)
    

In [140]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import timm
# import numpy as np
# from timm.models.vision_transformer import VisionTransformer, Block

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # ------------------------------
# # Custom Attention Module
# # ------------------------------
# class CustomAttention(nn.Module):
#     def __init__(self, attn, num_heads):
#         super().__init__()
#         self.num_heads = num_heads
#         self.scale = attn.scale
#         self.qkv = attn.qkv
#         self.attn_drop = attn.attn_drop
#         self.proj = attn.proj
#         self.proj_drop = attn.proj_drop

#     def forward(self, x):
#         B, N, C = x.shape
#         qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
#         q, k, v = qkv[0], qkv[1], qkv[2]
#         attn = (q @ k.transpose(-2, -1)) * self.scale
#         attn = attn.softmax(dim=-1)
#         attn = self.attn_drop(attn)
#         avg_attn = attn.mean(dim=1)
#         cls_token_attn = avg_attn[:, 0, :]
#         threshold = torch.quantile(cls_token_attn, 0.8, dim=-1, keepdim=True)
#         mask_attention = cls_token_attn >= threshold
#         x = (attn @ v).transpose(1, 2).reshape(B, N, C)
#         x = self.proj(x)
#         x = self.proj_drop(x)
#         return x, mask_attention

# # ------------------------------
# # Patch Drop Module
# # ------------------------------
# class PatchDropout(nn.Module):
#     def __init__(self):
#         super().__init__()

#     def forward(self, x, mask):
#         drop_rate = 0.0
#         B, N, C = x.shape
#         assert N == 198, "Expected 198 tokens (1 CLS + 1 DIST + 196 patches)"
#         cls_token = x[:, 0, :]
#         dist_token = x[:, 1, :]
#         patch_tokens = x[:, 2:, :]

#         similarity_scores = F.cosine_similarity(patch_tokens, cls_token.unsqueeze(1), dim=-1)
#         k = int(196 * drop_rate)
#         threshold_values, _ = torch.kthvalue(similarity_scores, 196 - k - 1, dim=1, keepdim=True)
#         keep_mask = similarity_scores < threshold_values
#         patch_tokens_filtered = patch_tokens[keep_mask].view(B, -1, C)

#         x_filtered = torch.cat([cls_token.unsqueeze(1), dist_token.unsqueeze(1), patch_tokens_filtered], dim=1)
#         return x_filtered

# # ------------------------------
# # Custom Transformer Block
# # ------------------------------
# class CustomBlock(nn.Module):
#     def __init__(self, block):
#         super().__init__()
#         self.norm1 = block.norm1
#         self.attn = CustomAttention(block.attn, block.attn.num_heads)
#         self.ls1 = block.ls1
#         self.drop_path1 = block.drop_path1
#         self.norm2 = block.norm2
#         self.mlp = block.mlp
#         self.ls2 = block.ls2
#         self.drop_path2 = block.drop_path2

#     def forward(self, x, apply_patch_drop=False):
#         x_residual = self.norm1(x)
#         x_residual, mask_attention = self.attn(x_residual)
#         x = x + self.drop_path1(self.ls1(x_residual))
#         x = x + self.drop_path2(self.ls2(self.mlp(self.norm2(x))))

#         if apply_patch_drop:
#             cls_token = x[:, 0:1]
#             dist_token = x[:, 1:2]
#             other_tokens = x[:, 2:]
#             mask = mask_attention[:, 1:]  # Apply on patch tokens only
#             kept = other_tokens[mask].view(x.size(0), -1, x.size(2))
#             x = torch.cat([cls_token, dist_token, kept], dim=1)

#         return x

# # ------------------------------
# # Custom ViT with Distillation Support
# # ------------------------------
# class CustomVisionTransformer(VisionTransformer):
#     def __init__(self, patch_drop_blocks=None, **kwargs):
#         super().__init__(**kwargs)
#         self.patch_drop_blocks = patch_drop_blocks if patch_drop_blocks is not None else []
#         self.patch_drop = PatchDropout()

#         # Replace blocks
#         self.blocks = nn.Sequential(*[
#             CustomBlock(block) if i in self.patch_drop_blocks else block
#             for i, block in enumerate(self.blocks)
#         ])

#     def forward(self, x, mask=None):
#         B = x.shape[0]
#         x = self.patch_embed(x)
#         cls_token = self.cls_token.expand(B, -1, -1)
#         dist_token = self.dist_token.expand(B, -1, -1)
#         x = torch.cat((cls_token, dist_token, x), dim=1)
#         x = self.pos_drop(x + self.pos_embed)

#         if mask is not None:
#             x = self.patch_drop(x, mask)

#         for block in self.blocks:
#             if isinstance(block, CustomBlock):
#                 x = block(x, apply_patch_drop=True)
#             else:
#                 x = block(x)

#         x = self.norm(x)
#         x_cls = self.head(x[:, 0])
#         x_dist = self.head_dist(x[:, 1])
#         return (x_cls + x_dist) / 2

# # ------------------------------
# # Instantiate Custom ViT
# # ------------------------------
# # Load pretrained DEiT-Tiny Distilled
# # pretrained_vit = timm.create_model('deit_tiny_patch16_224_distilled', pretrained=True).to(device)
# # pretrained_vit.eval()

# # Define which blocks to drop patches at
# patch_drop_block_number = []

# # Create Custom Model
# custom_vit = CustomVisionTransformer(
#     img_size=224,
#     patch_size=16,
#     embed_dim=192,
#     depth=12,
#     num_heads=3,
#     mlp_ratio=4,
#     qkv_bias=True,
#     norm_layer=nn.LayerNorm,
#     num_classes=1000,
#     patch_drop_blocks=patch_drop_block_number
# ).to(device)

# # FIX: Add positional embedding with correct token count (198 = cls + dist + 196)
# # custom_vit.pos_embed = nn.Parameter(torch.zeros(1, 198, custom_vit.embed_dim))

# # Load weights
# custom_vit.load_state_dict(pretrained_vit.state_dict(), strict=False)

# input_tensor = torch.randn(1, 3, 224, 224).to(device)
# mask = np.random.randint(1, 2, size=198)  # For PatchDropout
# output = custom_vit(input_tensor, mask=mask)
# print("Output shape:", output.shape)  # Should be [1, 1000]

In [141]:
# from qtorch.quant import Quantizer, quantizer
# from qtorch import FixedPoint
# # Define the fixed-point quantizer
# weight_quantizer = quantizer(forward_number=FixedPoint(16, 8), forward_rounding="nearest")
# act_quantizer = quantizer(forward_number=FixedPoint(16, 8), forward_rounding="nearest")

In [142]:
# import torch
# import torch.nn as nn

# # Define the QuantizedVisionTransformer class
# class QuantizedVisionTransformer(nn.Module):
#     def __init__(self, model, weight_quantizer, act_quantizer):
#         super(QuantizedVisionTransformer, self).__init__()
#         self.model = model
#         self.weight_quantizer = weight_quantizer
#         self.act_quantizer = act_quantizer
#         self.apply(self._quantize_weights)

#     def _quantize_weights(self, module):
#         if isinstance(module, nn.Linear):
#             module.weight.data = self.weight_quantizer(module.weight.data)
#             if module.bias is not None:
#                 module.bias.data = self.weight_quantizer(module.bias.data)
#         elif isinstance(module, nn.Conv2d):
#             module.weight.data = self.weight_quantizer(module.weight.data)
#             if module.bias is not None:
#                 module.bias.data = self.weight_quantizer(module.bias.data)

#     def forward(self, x, mask):
#         x = self.act_quantizer(x)
#         # for name, module in self.model.named_children():
#         #     x = module(x, mask) if 'block' in name else module(x)
#         #     if isinstance(module, (nn.Linear, nn.Conv2d)):
#         #         x = self.act_quantizer(x)
#         return self.model(x, mask)

# # Example usage:
# # Assume `vision_transformer_model` is your pretrained model
# # and `weight_quantizer` and `act_quantizer` are your quantization functions
# quantized_model = QuantizedVisionTransformer(custom_vit, weight_quantizer, act_quantizer)



In [143]:
# def check_quantization(model):
#     for name, param in model.named_parameters():
#         print(f"Parameter name: {name}")
#         print(f"Data type: {param.dtype}")
#         print(f"Values: {param.view(-1)[:10]}")  # Print the first 10 values
#         print(f"Storage size (bytes): {param.numel() * param.element_size()}")
#         print()



# # Check quantized weights and biases
# # check_quantization(quantized_model)

In [144]:
# output = quantized_model(input_tensor, mask=my_array)

In [145]:
def main_processing(image):
    resized_image =  resize_nearest_neighbor(image, (224, 224))
    resized_image = bgr_to_rgb(resized_image)
    # gray_image = rgb_to_gray(resized_image)
    resized_image = resized_image.copy()                                        
    image_tensor= transform_image(resized_image, device)
    mask = np.random.randint(1, 2, size=198) 
    
    return image_tensor, mask


In [146]:
def save_percentage(file_path, average_percent1, average_percent2, average_percent3 ,average_percent4, average, count):
    with open(file_path, 'a') as file:
        file.write(f"Epoch: Average percentage_patch_drop: { average_percent1}, { average_percent2}, { average_percent3}, { average_percent4}, average: {average}, count{count}\n")

In [147]:
def convert_to_canvas(img_folder, model, device):
    correct_predictions = 0
    total_images = 0
    processed_images = 0
    
    for subfolder_name in os.listdir(img_folder):
        subfolder_path = os.path.join(img_folder, subfolder_name)
        if os.path.isdir(subfolder_path):
            # Process each image in the subfolder
            for img_name in os.listdir(subfolder_path):
                img_path = os.path.join(subfolder_path, img_name)
                if os.path.isfile(img_path) and img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    # Load the image
                    image = cv2.imread(img_path)
                    if image is None:
                        print(f"Failed to load image: {img_path}")
                        continue

                    image_tensor, fast_mask = main_processing(image)
                    
                    # outputs = custom_vit( image_tensor, mask= fast_mask)
                    outputs= pretrained_vit(image_tensor)
                    predicted_idx = torch.argmax(outputs, dim=1).item()
                    match = re.search(r'val2/([^/]+)/', img_path)
                    if match:
                        class_name = match.group(1)
                    else:
                        class_name = None
                    
                    prediction_result = get_key_by_value(predicted_idx, class_name)
                    
                    if prediction_result is True:
                        correct_predictions += 1    
                    processed_images += 1
                    
                    # Print accuracy after processing every 50 images
                    if processed_images % 10 == 0:
                        # w= ( sum(scan)/ len(scan))
                        accuracy_so_far = (correct_predictions / processed_images) * 100
                        average_percent= 15_15_15_15
                        w=0
                        # average_percent = sum(patch_drop_percentage) / len(patch_drop_percentage)
                        # print("average percentage embedding_drop " , average_percent , len(patch_drop_percentage))
                        print(f"Accuracy after processing {processed_images} images: {accuracy_so_far:.2f}%")
                        save_average_percentage(txt_path, average_percent, accuracy_so_far, processed_images, w )
                    
                    
                    


In [148]:
# device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
# Example usage
print(device)

cuda:1


In [ ]:
convert_to_canvas(img_folder, pretrained_vit, device)

Accuracy after processing 10 images: 100.00%
Accuracy after processing 20 images: 95.00%
Accuracy after processing 30 images: 80.00%
Accuracy after processing 40 images: 70.00%
Accuracy after processing 50 images: 70.00%
Accuracy after processing 60 images: 68.33%
Accuracy after processing 70 images: 68.57%
Accuracy after processing 80 images: 71.25%
Accuracy after processing 90 images: 74.44%
Accuracy after processing 100 images: 75.00%
Accuracy after processing 110 images: 76.36%
Accuracy after processing 120 images: 77.50%
Accuracy after processing 130 images: 77.69%
Accuracy after processing 140 images: 78.57%
Accuracy after processing 150 images: 78.00%
Accuracy after processing 160 images: 78.12%
Accuracy after processing 170 images: 77.65%
Accuracy after processing 180 images: 77.78%
Accuracy after processing 190 images: 78.42%
Accuracy after processing 200 images: 76.00%
Accuracy after processing 210 images: 76.19%
Accuracy after processing 220 images: 76.36%
Accuracy after pro